# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook demonstrates loading and exploring the **FAIR^2** dataset using the `mlcroissant` library, utilizing schema information and structure defined in its Croissant metadata.

### Dataset Source
The dataset is defined by the following Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the Croissant schema and dataset metadata. This includes descriptive fields, available record sets, and the high-level structure of the dataset.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata via Croissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\n")
print("Description:")
print(metadata.description)


## 2. Data Overview

In this section, we review and print the available record sets, their `@id` identifiers, and associated fields/columns defined in the Croissant schema. All entities are referenced by their `@id`.

In [ ]:
# List all record sets in the dataset using their '@id'.
print('Record Sets defined in this dataset:')
for record_set in dataset.record_sets:
    print(f"- @id: {record_set['@id']}")
    # Print field and column @id's within each record set
    fields = record_set.get('field', [])
    # Fields may be dict if 1, list if many
    if isinstance(fields, dict):
        fields = [fields]
    if fields:
        print("  Fields @ids:")
        for f in fields:
            if isinstance(f, dict):
                print(f"    - {f['@id']}")
            else:
                print(f"    - {f}")
    columns = record_set.get('column', [])
    if isinstance(columns, dict):
        columns = [columns]
    if columns:
        print("  Columns @ids:")
        for c in columns:
            if isinstance(c, dict):
                print(f"    - {c['@id']}")
            else:
                print(f"    - {c}")
print("\n")

## 3. Data Extraction

We now extract records from each available record set. All references are by their `@id`. Data are loaded into DataFrames for analysis.

In [ ]:
# Gather all record set @id's as a list (for automation)
record_set_ids = [r['@id'] for r in dataset.record_sets]

dataframes = {}

# Load each record set's records into a pandas DataFrame
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)
    print(f"Loaded DataFrame for record set '{rs_id}'; shape: {dataframes[rs_id].shape}")

# For demonstration, show column names and first 5 rows for the first record set
if record_set_ids:
    first_rs_id = record_set_ids[0]
    print(f"\nColumns in record set '{first_rs_id}':")
    print(dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())
else:
    print("No record sets defined in this dataset.")

## 4. Exploratory Data Analysis (EDA)

We'll process values in a demonstrated numeric field from the first available record set. Analysis includes filtering, normalization, and grouping.

**Note:** Replace `<numeric_field_id>` and `<group_field>` with the actual `@id`s from the record set overview if required. (We attempt to infer these per schema; update as needed.)

In [ ]:
# Select a record set for analysis
if record_set_ids:
    rs_id = record_set_ids[0]
    df = dataframes[rs_id]
    print(f\"Using record set '@id': {rs_id}\")
    print("Available columns:", list(df.columns))
    
    # Try to find a numeric field (float or int) automatically
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    if numeric_field_id is None:
        print("No numeric field found for analysis in this record set.")
    else:
        print(f"Using numeric field '{numeric_field_id}' (@id) for EDA.")
        threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
        filtered_df = df[df[numeric_field_id] > threshold]

        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Attempt to group by a likely categorical field (first non-numeric field)
        group_field_id = None
        for col in df.columns:
            if not pd.api.types.is_numeric_dtype(df[col]) and df[col].nunique() > 1:
                group_field_id = col
                break
        if group_field_id:
            print(f"\nGrouping filtered data by '{group_field_id}' (@id):")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name=f"mean_{numeric_field_id}")
            display(grouped_df.head())
        else:
            print("No suitable categorical field for grouping found.")
else:
    print("No record sets found to perform EDA.")

## 5. Visualization

We visualize the distribution of the selected numeric field and a grouped aggregation if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and numeric_field_id is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of '{numeric_field_id}' (@id)")
    plt.xlabel(numeric_field_id)
    plt.show()

    if 'group_field_id' in locals() and group_field_id:
        # Plot group-wise means
        group_means = df.groupby(group_field_id)[numeric_field_id].mean()
        group_means.plot(kind='bar', figsize=(8,4))
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.title(f"Mean '{numeric_field_id}' by '{group_field_id}' (@id)")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("Visualization skipped: No numeric field available or no data loaded.")

## 6. Conclusion

Using the `mlcroissant` library, we've loaded the FAIR^2 dataset via its Croissant schema, inspected record sets and fields by their `@id` values, and performed basic EDA including filtering, normalization, grouping, and visualization on available fields. The structure and `@id`-first referencing ensure robustness and reproducibility for future automated dataset workflows. For further analysis or integration, consult the Croissant schema or dataset documentation.